# Temporal Overlap Pilot\n\nDoes detection of an asynchronous recurring figure depend on the **absolute duration** of overlap between its components, on the **proportion** of them that overlaps, or only on how far apart their **onsets** are?\n\nTwo tones can overlap by 50% while sharing very different amounts of time: 20 ms tones stepped by 10 ms share 10 ms, 40 ms tones stepped by 20 ms share 20. Whether that matters is the question. Nothing in this notebook assumes an answer.

In [ ]:
#@title setup
import sys, subprocess, importlib.util, json, math
from pathlib import Path
REF='564e21cd08cbdbd836d293ce3083dd69144d5b58'
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
try:
    from seqsfg import overlap as OV
except Exception:
    if bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        ROOT=Path('/content')/('seqsfg-'+REF[:12])
        if not ROOT.exists():
            subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)
            subprocess.run(['git','-C',str(ROOT),'checkout','--detach',REF],check=True)
    else:
        ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'seqsfg/overlap.py').exists()),None)
    for _m in [k for k in list(sys.modules) if k=='seqsfg' or k.startswith('seqsfg.')]: del sys.modules[_m]
    sys.path.insert(0,str(ROOT)); from seqsfg import overlap as OV

import numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
from seqsfg.config import validate
from seqsfg.stimulus import FIGURE, render_interval
ROOT=Path(OV.__file__).parents[1]
CFG, OCFG = OV.load_preset(ROOT/'overlap_pilot.json'); D = validate(CFG)
OV.check(CFG, OCFG)
OCT=np.log2(D.channel_freqs_hz); TICKS=[250,1000,4000]
C={'fig':'#c1272d','bg':'#c3c8cf','ink':'#1f2328','alt':'#1b6ca8','ok':'#2e7d32'}
plt.rcParams.update({'figure.dpi':115,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':.25,'font.size':9})
def play(x):
    pad=np.zeros(int(.05*CFG.sample_rate),dtype=np.float32)
    return Audio(np.clip(np.concatenate([pad,x]),-1,1),rate=CFG.sample_rate,normalize=False)
def head(t): display(Markdown(t))
SEED=12
print(f"{D.n_channels} channels, {D.channel_freqs_hz[0]:.0f}-{D.channel_freqs_hz[-1]:.0f} Hz, one per ERB")
print(f"{CFG.n_components} components, {CFG.n_elements} recurrences, background tones "
      f"{CFG.tone_dur_ms:.0f} ms, {CFG.tones_per_channel} per channel")
print(f"{D.mean_simultaneous:.1f} tones sounding at once; one tone calibrated to "
      f"{CFG.tone_level_db_spl:.0f} dB SPL puts the scene near "
      f"{CFG.tone_level_db_spl + 10*math.log10(D.mean_simultaneous):.0f} dB SPL")
print(f"scene {CFG.interval_dur_ms/1000:g} s, recurrence every "
      f"{CFG.iei_min_ms:.0f}-{CFG.iei_max_ms:.0f} ms, config {CFG.hash()}")

---\n## The conditions

In [ ]:
#@title the seven conditions
rows=[]
for t,s in OCFG.cells:
    g=OV.geometry(CFG,t,s,CFG.n_components); rows.append(g)
head("| cell | T (ms) | step (ms) | adjacent overlap | fraction | common to all 7 | "
     "total extent | envelope overlap | envelope fraction |\n"
     "|---|---|---|---|---|---|---|---|---|\n" + "\n".join(
     f"| `{OV.cell_name(g['tone_ms'],g['step_ms'])}` | {g['tone_ms']:.0f} | {g['step_ms']:.0f} | "
     f"{g['adjacent_overlap_ms']:.0f} ms | {g['adjacent_overlap_fraction']:.0%} | "
     f"{g['common_overlap_ms']:.0f} ms | {g['total_extent_ms']:.0f} ms | "
     f"{g['envelope_overlap_ms']:.2f} ms | {g['envelope_overlap_fraction']:.3f} |" for g in rows))
head("**Adjacent overlap** is what two *consecutive* components share. **Common overlap** is what "
     "all seven share at once, and it is zero in every asynchronous cell. The two are different "
     "quantities and the analysis keeps them apart.\n\n"
     "**Envelope overlap** weights the shared time by the raised-cosine gates. The ramp is "
     f"{CFG.ramp_ms:.0f} ms at both durations, so a {min(OCFG.durations):.0f} ms component is half "
     f"ramp and a {max(OCFG.durations):.0f} ms one a quarter. Notice that the two *equal absolute "
     "overlap* cells keep the same envelope overlap, while the two *equal fraction* cells do not.")
fig,ax=plt.subplots(1,3,figsize=(12.8,3.1))
x=np.arange(len(rows)); lab=[OV.cell_name(g['tone_ms'],g['step_ms']) for g in rows]
col=[C['fig'] if g['tone_ms']==20 else C['alt'] for g in rows]
for a,key,t in zip(ax,('adjacent_overlap_ms','adjacent_overlap_fraction','total_extent_ms'),
                   ('adjacent overlap (ms)','adjacent overlap (fraction)','total extent (ms)')):
    a.bar(x,[g[key] for g in rows],color=col); a.set_xticks(x)
    a.set_xticklabels(lab,rotation=45,ha='right',fontsize=7.5); a.set_title(t,fontsize=9.5)
ax[0].set_ylabel('ms')
plt.tight_layout(); plt.show()
print("red = 20 ms components, blue = 40 ms")

---\n## What the components actually do

In [ ]:
#@title onset and offset diagrams, one recurrence per cell
fig,ax=plt.subplots(2,4,figsize=(13.2,5.0))
for a,(t,s) in zip(ax.ravel(), list(OCFG.cells)+[(None,None)]*4):
    if s is None:
        a.axis('off'); continue
    g=OV.geometry(CFG,t,s,CFG.n_components)
    for i in range(CFG.n_components):
        a.add_patch(plt.Rectangle((i*s, i-0.36), t, 0.72, color=C['fig'] if t==20 else C['alt'], lw=0))
    a.plot([0,g['total_extent_ms']],[-1.1,-1.1],color=C['ink'],lw=1.4)
    a.text(g['total_extent_ms']/2,-1.9,f"extent {g['total_extent_ms']:.0f} ms",
           ha='center',fontsize=7.5,color=C['ink'])
    if g['adjacent_overlap_ms']>0:
        a.plot([s,t],[0.62,0.62],color=C['ok'],lw=2.6,solid_capstyle='butt')
        a.text((s+t)/2,1.15,f"{g['adjacent_overlap_ms']:.0f} ms",ha='center',fontsize=7.5,color=C['ok'])
    a.set_xlim(-12,300); a.set_ylim(-2.6,CFG.n_components+0.4)
    a.set_yticks([]); a.set_xlabel('ms',fontsize=8); a.tick_params(labelsize=7.5)
    a.set_title(f"{OV.cell_name(t,s)}   adjacent {g['adjacent_overlap_ms']:.0f} ms "
                f"({g['adjacent_overlap_fraction']:.0%})",fontsize=9)
plt.tight_layout(); plt.show()
head("Each bar is one component. The green mark is the overlap between the **first two**; every "
     "adjacent pair shares the same amount. Reading down a column shows what the same overlap "
     "fraction looks like at the two durations, and how far apart the extents are.")

In [ ]:
#@title the same thing measured on the rendered envelopes
from seqsfg.stimulus import tone_envelope
fig,ax=plt.subplots(1,2,figsize=(12.4,3.2))
for a,t in zip(ax, OCFG.durations):
    e=tone_envelope(CFG,t); tt=np.arange(e.size)/CFG.sample_rate*1000
    for s in sorted({s for tt_,s in OCFG.cells if tt_==t}):
        a.plot(tt, e, color=C['ink'], lw=1.0, alpha=.85 if s==0 else .25)
        a.plot(tt+s, e, color=C['fig'] if t==20 else C['alt'], lw=1.2,
               label=f"step {s:.0f} ms  (envelope overlap {OV.envelope_overlap(CFG,t,s)[1]:.2f})")
    a.set_title(f"{t:.0f} ms components, {CFG.ramp_ms:.0f} ms ramps",fontsize=9.5)
    a.set_xlabel('ms'); a.set_ylabel('amplitude'); a.legend(frameon=False,fontsize=7.5)
plt.tight_layout(); plt.show()
head("The grey curve is the first component, the coloured ones are the second at each step. "
     "Because the ramp is held at "
     f"{CFG.ramp_ms:.0f} ms, a geometric overlap that lands on the ramps is worth less envelope "
     "than one that lands on the plateau, which is why the envelope-weighted number is reported "
     "beside the geometric one rather than instead of it.")

---\n## A whole scene

In [ ]:
#@title what a whole scene looks like
cells=[(20.,0.),(20.,10.),(40.,20.),(40.,40.)]
fig,ax=plt.subplots(len(cells),2,figsize=(13.2,2.05*len(cells)),sharex=True,sharey=True)
for r,(t,s) in enumerate(cells):
    for c,present in enumerate((True,False)):
        iv=OV.build_interval(CFG,OCFG,D,SEED,t,s,present)
        tt=iv.onset*CFG.grid_ms/1000.; y=OCT[iv.channel]; m=iv.kind==FIGURE
        a=ax[r,c]
        a.scatter(tt[~m],y[~m],s=4,c=C['bg'],marker='s',lw=0)
        a.scatter(tt[m],y[m],s=20,c=C['fig'],marker='s',lw=0)
        a.set_xlim(0,CFG.interval_dur_ms/1000.); a.set_ylim(OCT[0]-.4,OCT[-1]+.4)
        a.set_yticks(np.log2(TICKS)); a.set_yticklabels([str(v) for v in TICKS])
        a.set_title(f"{OV.cell_name(t,s)}  -  {'pattern present' if present else 'no pattern'}",fontsize=9)
        if r==len(cells)-1: a.set_xlabel('time (s)')
    ax[r,0].set_ylabel('Hz')
plt.tight_layout(); plt.show()
head("Left: the seven components recur on the same pitches. Right: **the same tones**, same "
     "channels, same durations, same count, scattered so nothing recurs. The red marking on the "
     "left is a label in the schedule, not anything extra in the sound.")

---\n## Do the two classes contain the same tones?\n\nThey must, or the task becomes a duration-detection task.

In [ ]:
#@title the matching check, per frequency and per duration
rows=[]
for t,s in OCFG.cells:
    ok=0
    for j in range(12):
        a=OV.build_interval(CFG,OCFG,D,600+j,t,s,True); b=OV.build_interval(CFG,OCFG,D,600+j,t,s,False)
        ok+=int(OV.inventory(a)==OV.inventory(b))
    rows.append((OV.cell_name(t,s),ok,12))
print(f"{'cell':>10}{'inventories identical':>24}")
for n,k,tot in rows: print(f"{n:>10}{f'{k}/{tot}':>24}")
t,s=40.,20.
a=OV.build_interval(CFG,OCFG,D,7,t,s,True); b=OV.build_interval(CFG,OCFG,D,7,t,s,False)
inv_a, inv_b = OV.inventory(a), OV.inventory(b)
keys=sorted(set(inv_a)|set(inv_b))
fig,ax=plt.subplots(figsize=(12.4,3.0))
x=np.arange(len(keys))
ax.bar(x-.2,[inv_a.get(k,0) for k in keys],.4,color=C['fig'],label='pattern present')
ax.bar(x+.2,[inv_b.get(k,0) for k in keys],.4,color=C['ink'],label='no pattern')
ax.set_xticks(x[::2]); ax.set_xticklabels(
    [f"{D.channel_freqs_hz[c]:.0f} Hz\n{u*CFG.grid_ms:.0f} ms" for c,u in keys][::2],fontsize=6.5)
ax.set_ylabel('tones'); ax.legend(frameon=False,fontsize=8.5,ncol=2)
ax.set_title(f"{OV.cell_name(t,s)}: every (frequency, duration) bin, both classes",fontsize=9.5)
plt.tight_layout(); plt.show()
head("Matching is checked per **(frequency, duration)** bin, not on the total count. If the "
     "20 ms components appeared only on pattern-present trials, the task would be a "
     "duration-detection task in disguise.")

---\n## Listen

In [ ]:
#@title listen: full mixtures, pattern present and absent
head("Full mixtures, exactly what a listener hears. Same seed throughout, so the figure is the "
     "same seven pitches in every cell.")
for t,s in OCFG.cells:
    g=OV.geometry(CFG,t,s,CFG.n_components)
    head(f"**`{OV.cell_name(t,s)}`** &nbsp; {t:.0f} ms components, {s:.0f} ms apart, adjacent "
         f"overlap {g['adjacent_overlap_ms']:.0f} ms ({g['adjacent_overlap_fraction']:.0%}), "
         f"extent {g['total_extent_ms']:.0f} ms")
    for present in (True,False):
        head(f"&nbsp;&nbsp;{'pattern present' if present else 'no pattern'}:")
        display(play(OV.render(CFG,OCFG,D,OV.build_interval(CFG,OCFG,D,SEED,t,s,present))))

In [ ]:
#@title ILLUSTRATION ONLY: the figure with the cloud removed
head("**This is not a stimulus.** No listener ever hears this. The cloud is stripped out so the "
     "recurring pattern can be heard on its own; it is here to make the diagrams audible.")
for t,s in ((20.,10.),(40.,20.),(40.,40.)):
    iv=OV.build_interval(CFG,OCFG,D,SEED,t,s,True)
    m=iv.kind==FIGURE; bare=iv.copy()
    for a_ in ('onset','channel','phase','kind','element','component','dur'):
        v=getattr(iv,a_)
        if v is not None: setattr(bare,a_,np.asarray(v)[m])
    head(f"&nbsp;&nbsp;`{OV.cell_name(t,s)}` figure alone")
    display(play(render_interval(CFG,bare,D)))

---\n## The analysis

In [ ]:
#@title the analysis, exercised on a SIMULATED listener
head("### This cell contains no data.\n"
     "It runs a simulated responder through the whole pipeline so the analysis can be read "
     "before anyone is tested. **Every number and bar below is synthetic.** Replace the session "
     "path with a real one to see real results.")
import tempfile, warnings
small=OV.OverlapConfig(**{**OCFG.to_dict(),'trials_per_cell':10,'practice_trials':4})
with tempfile.TemporaryDirectory() as tmp:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sdir=OV.OverlapRunner(CFG,small,tmp,audio=False,auto=150.0).run(code='SIMULATED',session_index=1)
    import csv
    rows=[r for r in csv.DictReader(open(Path(sdir)/'trials.csv')) if r['block']=='main']
    text=OV.analyse([sdir])
print(text)
cells=list(OCFG.cells)
dp=[]; lo=[]; hi=[]
counts={OV.cell_name(t,s):OV._counts([r for r in rows if r['variant']==OV.cell_name(t,s)]) for t,s in cells}
boots=OV._boot(counts,2000)
for t,s in cells:
    k=OV.cell_name(t,s); dp.append(OV._dp(counts[k]))
    b=np.array([OV._dp(x[k]) for x in boots]); b=b[np.isfinite(b)]
    lo.append(np.quantile(b,.025)); hi.append(np.quantile(b,.975))
fig,ax=plt.subplots(1,2,figsize=(12.8,3.4))
x=np.arange(len(cells)); col=[C['fig'] if t==20 else C['alt'] for t,_ in cells]
ax[0].bar(x,dp,color=col)
ax[0].errorbar(x,dp,yerr=[np.maximum(0,np.array(dp)-lo),np.maximum(0,np.array(hi)-np.array(dp))],
               fmt='none',ecolor=C['ink'],lw=1)
ax[0].set_xticks(x); ax[0].set_xticklabels([OV.cell_name(t,s) for t,s in cells],rotation=45,
                                           ha='right',fontsize=7.5)
ax[0].set_ylabel("d' (SIMULATED)"); ax[0].axhline(0,color='k',lw=.6)
ax[0].set_title("per cell -- synthetic",fontsize=9.5)
ov=[OV.geometry(CFG,t,s,CFG.n_components)['adjacent_overlap_ms'] for t,s in cells]
for dur,mk in ((20.,'o'),(40.,'s')):
    m=[i for i,(t,_) in enumerate(cells) if t==dur]
    ax[1].plot([ov[i] for i in m],[dp[i] for i in m],mk+'-',
               c=C['fig'] if dur==20 else C['alt'],label=f'{dur:.0f} ms components')
ax[1].set_xlabel('adjacent overlap (ms)'); ax[1].set_ylabel("d' (SIMULATED)")
ax[1].legend(frameon=False,fontsize=8.5); ax[1].set_title("against absolute overlap -- synthetic",fontsize=9.5)
plt.tight_layout(); plt.show()
head("Two curves that fall together against **absolute** overlap would favour an absolute "
     "account; two that fall together against the **fraction** would favour a proportional one. "
     "With ten trials a cell neither is decidable, which is what the intervals are showing.")

---\n## What this pilot cannot tell you

* **Step, duration and overlap are one quantity, not three.** overlap = max(0, T - step), so they
  cannot be entered as independent predictors of anything. Any model has to fix one.
* **The longer component carries more energy.** At equal amplitude a 40 ms tone has about 3 dB
  more than a 20 ms one. The synchronous cell at each duration is where a general duration
  benefit would appear, but it does not remove the confound, and normalising the mixture RMS
  would not either -- that equalises the scene, not the component.
* **Ten trials a cell is a feasibility sample.** It is not powered to separate an absolute-overlap
  account from a proportional one. A difference whose interval contains zero is not evidence that
  the two are equivalent.
* **The ladder is not a threshold.** Nothing here should be fitted with a monotonic psychometric
  function and read off as a limit.
* **Baseline-adjusted contrasts are exploratory.** Subtracting the synchronous cell does not
  isolate overlap: the reference differs from its cell in step, in extent and in common overlap
  at the same time.
* **The two synchronous cells carry an acoustic cue.** Seven tones starting together are a level
  event a scattered cloud does not have, and no arrangement of the same tones avoids it. The
  audit measures it (learnt observer d' near +1.2 and +1.5); the five asynchronous cells sit at
  chance across seeds. Read the synchronous cells as a manipulation check.